# 🛡️ SecOpsOps — RL Training on SOC Environment

**Meta PyTorch OpenEnv Hackathon — Grand Finale**

This notebook trains a small LLM to act as a SOC analyst using GRPO (Group Relative Policy Optimization) via HuggingFace TRL.

Environment: https://itzrealmee-secopsops.hf.space  
GitHub: https://github.com/gggsiw/secopsops

In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers datasets peft accelerate requests matplotlib

In [ ]:
import requests, json, random, re
import matplotlib.pyplot as plt
from collections import defaultdict

ENV_URL = "https://itzrealmee-secopsops.hf.space"

def env_reset(task="easy"):
    r = requests.post(f"{ENV_URL}/reset", json={"task_name": task})
    return r.json()

def env_step(task, action_type, query=None, content=None, priority=None):
    payload = {
        "task_name": task,
        "action": {
            "action_type": action_type,
            "query": query,
            "content": content,
            "priority": priority
        }
    }
    r = requests.post(f"{ENV_URL}/step", json=payload)
    return r.json()

# Test connection
health = requests.get(f"{ENV_URL}/health").json()
print("✅ Environment connected:", health)

In [ ]:
# --- Load model with Unsloth ---
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"✅ Model loaded: {MODEL_NAME}")

In [ ]:
# --- Build training dataset from environment rollouts ---

SYSTEM_PROMPT = """You are a SOC analyst AI. Analyze the alert and respond with:
THINK: <1-2 sentence reasoning>
ACTION: <one of: investigate | query_logs | block_ip | escalate | close | report | create_ticket | query_siem>

Rules:
- false_positive → close
- malware/high → block_ip
- exfiltration/high → block_ip (data leaving NOW)
- lateral_movement/high → escalate
- login/high → escalate
- login/medium → query_logs FIRST then block_ip
- phishing → investigate/query_logs THEN report"""

# Optimal action map for dataset generation
OPTIMAL = {
    ("malware", "high"): "block_ip",
    ("false_positive", "low"): "close",
    ("false_positive", "medium"): "close",
    ("false_positive", "high"): "investigate",
    ("login", "high"): "escalate",
    ("login", "medium"): "query_logs",
    ("login", "low"): "investigate",
    ("phishing", "low"): "query_logs",
    ("phishing", "medium"): "query_logs",
    ("phishing", "high"): "escalate",
    ("lateral_movement", "high"): "escalate",
    ("exfiltration", "high"): "block_ip",
}

def make_prompt(alert, history):
    hist_str = " → ".join(history[-3:]) if history else "none"
    return (
        f"Alert Type: {alert['type']}\n"
        f"Severity: {alert['severity']}\n"
        f"Description: {alert['description']}\n"
        f"Source IP: {alert['source_ip']}\n"
        f"History: {hist_str}"
    )

def make_ideal_response(alert):
    action = OPTIMAL.get((alert['type'], alert['severity']), "investigate")
    think_map = {
        "block_ip": f"This is a confirmed {alert['type']} threat with {alert['severity']} severity. Immediate containment required.",
        "close": "This is a known false positive. Safe to close without further action.",
        "escalate": f"High severity {alert['type']} event. This requires Tier-2 escalation immediately.",
        "query_logs": f"Medium severity {alert['type']}. Need to review logs before taking action.",
        "investigate": f"Low severity {alert['type']}. Gather more context before deciding.",
        "report": "Investigation complete. Filing formal incident report.",
    }
    think = think_map.get(action, "Analyzing alert context carefully.")
    return f"THINK: {think}\nACTION: {action}"

# Collect training examples from all 3 tasks
training_data = []
for task_name in ["easy", "medium", "hard"]:
    obs = env_reset(task_name)
    history = []
    for alert in obs.get("alerts", []):
        prompt = make_prompt(alert, history)
        response = make_ideal_response(alert)
        training_data.append({
            "prompt": prompt,
            "response": response,
            "task": task_name,
            "alert_type": alert["type"],
            "severity": alert["severity"],
        })
        action = OPTIMAL.get((alert["type"], alert["severity"]), "investigate")
        history.append(action)

print(f"✅ Training examples generated: {len(training_data)}")
print("\nSample:")
print("PROMPT:", training_data[0]["prompt"])
print("RESPONSE:", training_data[0]["response"])

In [ ]:
# --- Reward function for GRPO/PPO training ---

VALID_ACTIONS = ["investigate", "query_logs", "block_ip", "escalate",
                 "close", "report", "create_ticket", "query_siem"]

def extract_action(text: str) -> str | None:
    """Extract action from model output."""
    for line in reversed(text.strip().split("\n")):
        if "ACTION:" in line.upper():
            parts = line.split(":", 1)
            if len(parts) > 1:
                action = parts[1].strip().lower().split()[0] if parts[1].strip() else None
                if action in VALID_ACTIONS:
                    return action
    return None

def compute_reward_from_env(task_name: str, alert_dict: dict, action: str, history: list) -> float:
    """Get reward from the live environment."""
    try:
        env_reset(task_name)  # Fresh episode
        # Simulate history by stepping through dummy actions first
        result = env_step(task_name, action, query=alert_dict.get("source_ip"))
        return result.get("reward", {}).get("final_score", 0.0)
    except:
        return 0.0

print("✅ Reward function ready")

In [ ]:
# --- SFT Training with TRL ---
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def format_for_sft(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_for_sft)

sft_config = SFTConfig(
    output_dir="/tmp/secopsops-sft",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_steps=50,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print("🚀 Starting SFT training...")
train_result = trainer.train()
print("✅ Training complete!")
print(train_result.metrics)

In [ ]:
# --- Evaluate trained model on environment ---
from unsloth import FastLanguageModel as FLM

FLM.for_inference(model)

def model_predict(alert_dict, history):
    prompt = make_prompt(alert_dict, history)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded

# Evaluate on all tasks
eval_results = {}
for task_name in ["easy", "medium", "hard"]:
    obs = env_reset(task_name)
    history = []
    task_rewards = []
    
    for alert in obs["alerts"]:
        output = model_predict(alert, history)
        action = extract_action(output) or "investigate"
        result = env_step(task_name, action, query=alert.get("source_ip"))
        reward = result.get("reward", {}).get("final_score", 0.0)
        task_rewards.append(reward)
        history.append(action)
        print(f"  [{task_name}] alert={alert['type']}/{alert['severity']} action={action} reward={reward:.2f}")
    
    eval_results[task_name] = task_rewards
    print(f"  → {task_name} avg: {sum(task_rewards)/len(task_rewards):.4f}\n")

In [ ]:
# --- Plot reward curves ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SecOpsOps — Reward Curves by Task (Post-Training)', fontsize=14, fontweight='bold')

colors = {'easy': '#2ea043', 'medium': '#f0883e', 'hard': '#f85149'}
labels = {'easy': 'Easy (Malware)', 'medium': 'Medium (Brute Force)', 'hard': 'Hard (APT Chain)'}

for i, (task, rewards) in enumerate(eval_results.items()):
    ax = axes[i]
    steps = list(range(1, len(rewards)+1))
    ax.bar(steps, rewards, color=colors[task], alpha=0.8, edgecolor='black')
    ax.axhline(y=sum(rewards)/len(rewards), color='red', linestyle='--', label=f'Avg: {sum(rewards)/len(rewards):.2f}')
    ax.set_title(labels[task])
    ax.set_xlabel('Alert Step')
    ax.set_ylabel('Reward (0-1)')
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Reward curves saved to reward_curves.png")

In [ ]:
# --- Training loss curve (from SFT logs) ---
log_history = trainer.state.log_history
losses = [(entry['step'], entry['loss']) for entry in log_history if 'loss' in entry]

if losses:
    steps, loss_vals = zip(*losses)
    plt.figure(figsize=(8, 4))
    plt.plot(steps, loss_vals, color='#58a6ff', linewidth=2)
    plt.fill_between(steps, loss_vals, alpha=0.2, color='#58a6ff')
    plt.title('SecOpsOps SFT Training Loss', fontsize=13, fontweight='bold')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Training loss curve saved")